# P119 — WaveNet: un modelo generativo de audio en crudo

## 1. Título y paper

**Paper:** *WaveNet: A Generative Model for Raw Audio*  
**Autoría:** Aäron van den Oord, Sander Dieleman, Heiga Zen, Karen Simonyan, Oriol Vinyals, Alex Graves, Nal Kalchbrenner, Andrew Senior, Koray Kavukcuoglu  
**Año y venue:** 2016 · arXiv:1609.03499  
**Nivel:** L3 · **Motor:** `wavenet`  
**Ficha completa:** [`P119_wavenet`](../../papers/foundational/P119_wavenet/README.md)

**Hito:** Genera la forma de onda muestra a muestra con convoluciones causales dilatadas, y cierra la brecha de naturalidad que arrastraba la síntesis de voz.

- [arXiv:1609.03499](https://arxiv.org/abs/1609.03499)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Modelar audio directamente exige un contexto de miles de muestras: a 16 kHz, un segundo son 16 000 valores. Una convolución normal necesitaría miles de capas para verlo, y una recurrente no puede entrenarse en paralelo sobre esa longitud.
2. Ejecutar una implementación mínima de la propuesta: Convoluciones causales con dilatación que se duplica por capa: el campo receptivo crece de forma exponencial con la profundidad. Más cuantización μ-law para que 256 niveles basten sin que la voz suene rota.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P04


## 4. Intuición

Para modelar un segundo de audio hay que ver 16 000 muestras. Con convoluciones normales harían falta 16 000 capas. Dilatando —saltando huecos que se duplican por capa— bastan 14.


## 5. Concepto mínimo

```text
Campo receptivo con núcleo 2:
  sin dilatar   : 1 + Σ 1        → crece LINEAL con la profundidad
  dilatado 2^i  : 1 + Σ 2^i      → crece EXPONENCIAL

Causal: la salida en t solo depende de entradas ≤ t
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('wavenet', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántas capas dilatadas hacen falta para un segundo a 16 kHz?
2. ¿Y sin dilatar?
3. ¿Puede el modelo mirar el futuro?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('wavenet', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('wavenet', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

**14 capas** dilatadas frente a **15 999** sin dilatar: un factor de **1 143×**. Y la convolución es causal — un impulso en la muestra 12 solo afecta a las muestras **12–19**, ninguna anterior. Sin esa restricción el modelo se entrenaría mirando el futuro y no serviría para generar.


## 10. Comentario pedagógico

La segunda idea es la cuantización μ-law: el 10 % central de la amplitud —donde vive la voz— se lleva **109** de los 256 códigos con μ-law y solo **26** con la lineal. Gastar resolución donde hay señal es lo que permite que 8 bits basten, y es una decisión de ingeniería de audio, no de aprendizaje automático.


## 11. Error o anti-patrón deliberado

Anti-patrón: creer que campo receptivo grande equivale a contexto usado.


In [ ]:
print('El campo receptivo dice que PUEDE ver el modelo, no que usa.')
print('Que alcance un segundo no significa que el segundo entero influya.')
print('Medir la influencia real exige ablaciones, no aritmetica de capas.')

## 12. Corrección

La geometría de las dilataciones:


In [ ]:
r = run_paper_lab('wavenet', seed=3)['result']
for fila in r['crecimiento_del_campo_receptivo']:
    print(fila)
print('para un segundo:', r['capas_para_cubrir_un_segundo'])
print('cuantizacion:', r['cuantizacion'])

## 13. Desafío guiado

Explica por qué la causalidad es obligatoria en un modelo generativo autorregresivo y qué se rompe exactamente si se entrena con una convolución no causal.


In [ ]:
r = run_paper_lab('wavenet', seed=3)['result']
show(r)

## 14. Desafío autónomo

Calcula el campo receptivo de una red convolucional que uses y compáralo con la longitud real de tus entradas. Decide si sobra o falta profundidad.


## 15. Evidencia de aprendizaje

Guarda el cálculo y la conclusión sobre tu red.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P119_wavenet/README.md) · evaluación formal: [`assessments/papers/P119_wavenet.md`](../../assessments/papers/P119_wavenet.md)


## 16. Cierre

El audio en crudo es viable; lo que falta es guiarlo con texto, que es la clase siguiente.


## 17. Conexión con el siguiente hito

- P122

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
